In [45]:
import pandas as pd
import re
import nltk
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from nltk.stem import PorterStemmer


from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

### Get all unique values for spell.csv the "type coulmn"

In [16]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize



df = pd.read_csv("Spells.csv")
print(df.head())
print(df.columns.tolist())

   Spell ID       Incantation                        Spell Name  \
0         1             Accio                   Summoning Charm   
1         2         Aguamenti                Water-Making Spell   
2         3  Alarte Ascendare  Launch an object up into the air   
3         4         Alohomora                   Unlocking Charm   
4         5     Arania Exumai            Spider repelling spell   

                  Effect     Light  
0      Summons an object       NaN  
1         Conjures water  Icy blue  
2  Rockets target upward       Red  
3         Unlocks target      Blue  
4         Repels spiders      Blue  
['Spell ID', 'Incantation', 'Spell Name', 'Effect', 'Light']


# Show all unique spell names

In [17]:
# Show all unique spell names
unique_spell_names = df["Spell Name"].dropna().unique()

print(unique_spell_names)
print(f"\nTotal unique spell names: {len(unique_spell_names)}")

['Summoning Charm' 'Water-Making Spell' 'Launch an object up into the air'
 'Unlocking Charm' 'Spider repelling spell' 'Slowing Charm'
 'Killing Curse' 'Exploding Charm' 'Brackium Emendo' 'Cistem Aperio'
 'Locking Spell' 'Blasting Curse' 'Cruciatus Curse' 'Severing Charm'
 'Dissendium' 'Engorgement Charm' 'Episkey' 'Patronus Charm'
 'Disarming Charm' 'Expulso Curse' 'General Counter-Spell'
 'Human Presence Revealing Spell' 'Freezing Charm' 'Impediment Jinx'
 'Imperius Curse' 'Incarcerous Spell' 'Fire-Making Spell' 'Levicorpus'
 'Locomotion Charm' 'Leg-Locker Curse' 'Wand-Lighting Charm'
 'Lumos Maxima' 'Lumos Solem Spell' 'Muffliato Charm'
 'Wand-Extinguishing Charm' 'Memory Charm' 'Oculus Reparo' 'Oppugno Jinx'
 'Peskipiksi Pesternomi' 'Full Body-Bind Curse' 'Piertotum Locomotor'
 'Portus' 'Reverse Spell' 'Shield Charm' 'Protego Maxima'
 'Protego totalum' 'Shrinking Charm' 'Revulsion Jinx' 'Mending Charm'
 'Repello Inimicum' 'Muggle-Repelling Charm' 'Revelio Charm'
 'Tickling Charm' '

In [18]:
print(df.isnull().sum())

Spell ID        0
Incantation     0
Spell Name      0
Effect          0
Light          21
dtype: int64


In [51]:
# Combine text fields
df["text"] = df["Spell Name"] + " " + df["Effect"]

# Lowercase
df["text"] = df["text"].str.lower()

# Remove punctuation
df["text"] = df["text"].apply(
    lambda x: re.sub(r"[^a-zA-Z\s]", "", x)
)

# Tokenization
df["tokens"] = df["text"].apply(word_tokenize)

# Standard English stopwords
stop_words = set(stopwords.words("english"))

# Harry Potter / dataset-specific stopwords
custom_stopwords = {
    "spell",
    "charm",
    "curse",
    "target",
    "object",
    "force",
    "forc",
    "maximum",
    "maxima",
    "wand",
    "jinx"
}

# Combine both sets
all_stopwords = stop_words.union(custom_stopwords)

# Remove stopwords
df["tokens"] = df["tokens"].apply(
    lambda words: [w for w in words if w not in all_stopwords]
)

stemmer = PorterStemmer()

df["tokens"] = df["tokens"].apply(
    lambda words: [stemmer.stem(w) for w in words]
)


# View results
print(df[["Spell Name", "tokens"]].head(10))

                         Spell Name                             tokens
0                   Summoning Charm                   [summon, summon]
1                Water-Making Spell          [watermak, conjur, water]
2  Launch an object up into the air      [launch, air, rocket, upward]
3                   Unlocking Charm                   [unlock, unlock]
4            Spider repelling spell     [spider, repel, repel, spider]
5                     Slowing Charm  [slow, slow, stop, target, veloc]
6                     Killing Curse           [kill, instantan, death]
7                   Exploding Charm            [explod, small, explos]
8                   Brackium Emendo     [brackium, emendo, mend, bone]
9                     Cistem Aperio      [cistem, aperio, open, chest]


### Making a dictionary

In [52]:
dictionary = Dictionary(df["tokens"])
dictionary.filter_extremes(
    no_below=2,
    no_above=0.7
)
corpus = [dictionary.doc2bow(text) for text in df["tokens"]]

### Small model

In [53]:
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=3,
    random_state=42,
    passes=20
)

In [54]:
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}: {topic}")

Topic 0: 0.178*"object" + 0.145*"conjur" + 0.144*"summon" + 0.111*"snake" + 0.078*"mend" + 0.078*"explos" + 0.078*"turn" + 0.044*"water" + 0.012*"protego" + 0.011*"wandlight"
Topic 1: 0.184*"reveal" + 0.122*"protego" + 0.099*"secret" + 0.099*"produc" + 0.099*"lumo" + 0.099*"anim" + 0.058*"water" + 0.056*"shield" + 0.015*"summon" + 0.014*"repel"
Topic 2: 0.193*"repel" + 0.104*"spell" + 0.104*"forc" + 0.104*"movement" + 0.104*"stop" + 0.104*"wandlight" + 0.060*"shield" + 0.034*"protego" + 0.015*"turn" + 0.015*"explos"


In [55]:
# Store dominant topic for each spell
dominant_topics = []

# Store full topic probabilities
topic_distributions = []

for bow in corpus:

    # Get topic probabilities
    topic_probs = lda_model.get_document_topics(bow)

    # Save full probability distribution
    topic_distributions.append(topic_probs)

    # Get dominant topic
    dominant_topic = max(topic_probs, key=lambda x: x[1])

    dominant_topics.append(dominant_topic[0])

# Add dominant topic column to dataframe
df["Dominant Topic"] = dominant_topics

In [56]:
topic_labels = {
    0: "Combat/Force Magic",
    1: "Protective Utility Magic",
    2: "Revealing/Conjuration Magic"
}

df["Topic Label"] = df["Dominant Topic"].map(topic_labels)

In [57]:
print(df[["Spell Name", "Dominant Topic", "Topic Label"]].head(20))

                          Spell Name  Dominant Topic  \
0                    Summoning Charm               0   
1                 Water-Making Spell               0   
2   Launch an object up into the air               0   
3                    Unlocking Charm               0   
4             Spider repelling spell               2   
5                      Slowing Charm               2   
6                      Killing Curse               0   
7                    Exploding Charm               0   
8                    Brackium Emendo               0   
9                      Cistem Aperio               0   
10                     Locking Spell               0   
11                    Blasting Curse               0   
12                   Cruciatus Curse               0   
13                    Severing Charm               0   
14                        Dissendium               1   
15                 Engorgement Charm               0   
16                           Episkey            